# ICD coding pilot: chapter-routed vs specialty-routed Mixture-of-Experts

**Before running:** Settings (right panel) → *Accelerator* = **GPU T4 x2** (or P100), *Internet* = **On** (needs a phone-verified Kaggle account). Then **Run All**. Total time ≈ 2.5–3 h on a T4.

1. Downloads open data: CodiEsp (Zenodo, CC BY 4.0), MTSamples (Hugging Face), MIMIC-IV Demo (PhysioNet, open).
2. `mimic_demo.py`: hospital service vs ICD chapter on the MIMIC-IV demo.
3. `prepare.py`: pseudo-specialty classifier (MTSamples) + frozen Bio_ClinicalBERT embeddings.
4. `run_moe.py`: 10 routing variants × 5 seeds on frozen embeddings (minutes).
5. `finetune.py`: 5 key variants × 3 seeds with Bio_ClinicalBERT fine-tuned end to end (the slow GPU part).

Results land in `/kaggle/working/VIP/pilot/out/` and are zipped at the end for download.

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/VIP/pilot/out /kaggle/working/VIP/data/mimic-demo
cd /kaggle/working/VIP/data
curl -sL -o codiesp.zip 'https://zenodo.org/api/records/3837305/files/codiesp.zip/content' && unzip -q -o codiesp.zip -d codiesp
curl -sL -o mtsamples.csv https://huggingface.co/datasets/harishnair04/mtsamples/resolve/main/mtsamples.csv
for t in admissions diagnoses_icd services d_icd_diagnoses patients; do
  curl -sL -o mimic-demo/$t.csv.gz https://physionet.org/files/mimic-iv-demo/2.2/hosp/$t.csv.gz
done
pip install -q tabulate
ls -la . mimic-demo

In [ ]:
%%writefile /kaggle/working/VIP/pilot/common.py
"""Shared helpers: ICD-9/10 chapter mapping, specialty grouping, paths."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"
OUT = ROOT / "pilot" / "out"
OUT.mkdir(parents=True, exist_ok=True)
CODIESP = DATA / "codiesp" / "final_dataset_v4_to_publish"

# ICD-10-CM chapters: (first code, last code, short name). Compared on the first 3 chars.
ICD10_CHAPTERS = [
    ("A00", "B99", "Infectious"), ("C00", "D49", "Neoplasms"), ("D50", "D89", "Blood/Immune"),
    ("E00", "E89", "Endocrine"), ("F01", "F99", "Mental"), ("G00", "G99", "Nervous"),
    ("H00", "H59", "Eye"), ("H60", "H95", "Ear"), ("I00", "I99", "Circulatory"),
    ("J00", "J99", "Respiratory"), ("K00", "K95", "Digestive"), ("L00", "L99", "Skin"),
    ("M00", "M99", "Musculoskeletal"), ("N00", "N99", "Genitourinary"), ("O00", "O9A", "Pregnancy"),
    ("P00", "P96", "Perinatal"), ("Q00", "Q99", "Congenital"), ("R00", "R99", "Symptoms"),
    ("S00", "T88", "Injury"), ("U00", "U85", "Special"), ("V00", "Y99", "External"),
    ("Z00", "Z99", "Health status"),
]
CHAPTER_NAMES = [c[2] for c in ICD10_CHAPTERS]

# ICD-9-CM numeric ranges mapped onto the ICD-10 chapter names above, so both versions share one axis.
ICD9_RANGES = [
    (1, 139, "Infectious"), (140, 239, "Neoplasms"), (240, 279, "Endocrine"), (280, 289, "Blood/Immune"),
    (290, 319, "Mental"), (320, 359, "Nervous"), (360, 379, "Eye"), (380, 389, "Ear"),
    (390, 459, "Circulatory"), (460, 519, "Respiratory"), (520, 579, "Digestive"),
    (580, 629, "Genitourinary"), (630, 679, "Pregnancy"), (680, 709, "Skin"),
    (710, 739, "Musculoskeletal"), (740, 759, "Congenital"), (760, 779, "Perinatal"),
    (780, 799, "Symptoms"), (800, 999, "Injury"),
]


def icd10_chapter(code: str) -> str | None:
    c = code.replace(".", "").upper()[:3]
    for lo, hi, name in ICD10_CHAPTERS:
        if lo <= c <= hi:
            return name
    return None


def icd9_chapter(code: str) -> str | None:
    c = code.replace(".", "").upper()
    if c.startswith("V"):
        return "Health status"
    if c.startswith("E"):
        return "External"
    try:
        n = int(c[:3])
    except ValueError:
        return None
    for lo, hi, name in ICD9_RANGES:
        if lo <= n <= hi:
            return name
    return None


def icd_chapter(code: str, version: int = 10) -> str | None:
    return icd10_chapter(code) if version == 10 else icd9_chapter(code)


# MTSamples specialty -> MIMIC-style service group. Labels that are note types
# (Consult, SOAP, Discharge Summary...) or too small are dropped (absent from this map).
MTSAMPLES_TO_SERVICE = {
    "General Medicine": "MED",
    "Cardiovascular / Pulmonary": "CMED",
    "Surgery": "SURG",
    "Orthopedic": "ORTHO",
    "Neurology": "NMED",
    "Neurosurgery": "NSURG",
    "Gastroenterology": "GI",
    "Obstetrics / Gynecology": "GYN",
    "Urology": "GU",
    "Nephrology": "GU",
    "Hematology - Oncology": "OMED",
    "Psychiatry / Psychology": "PSYCH",
}
SERVICES = sorted(set(MTSAMPLES_TO_SERVICE.values()))


if __name__ == "__main__":
    assert icd10_chapter("s22.49xa") == "Injury"
    assert icd10_chapter("D50.9") == "Blood/Immune" and icd10_chapter("d49") == "Neoplasms"
    assert icd10_chapter("O9A.1") == "Pregnancy" and icd10_chapter("t88.7") == "Injury"
    assert icd9_chapter("4019") == "Circulatory" and icd9_chapter("V5861") == "Health status"
    assert icd9_chapter("E8490") == "External" and icd9_chapter("0389") == "Infectious"
    print("ok")


In [ ]:
%%writefile /kaggle/working/VIP/pilot/mimic_demo.py
"""Experiment 1: how does MIMIC hospital service relate to ICD chapter? (MIMIC-IV Demo, 275 admissions)"""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import mutual_info_score

from common import DATA, OUT, icd_chapter

d = DATA / "mimic-demo"
services = pd.read_csv(d / "services.csv.gz", parse_dates=["transfertime"])
dx = pd.read_csv(d / "diagnoses_icd.csv.gz")

# First service of each admission = the specialty label a router would see.
first = services.sort_values("transfertime").groupby("hadm_id").curr_service.first()
n_services = services.groupby("hadm_id").curr_service.nunique()
dx["chapter"] = [icd_chapter(c, v) for c, v in zip(dx.icd_code, dx.icd_version)]
dx = dx.dropna(subset=["chapter"])
dx = dx[dx.hadm_id.isin(first.index)]
chapters = dx.groupby("hadm_id").chapter.apply(set)
first = first.loc[chapters.index]

lines = [f"Admissions with service + diagnoses: {len(first)}"]
lines.append(f"Share whose first service is MED (general medicine): {(first == 'MED').mean():.1%}")
lines.append(f"Admissions that change service during stay: {(n_services.loc[first.index] > 1).mean():.1%}")
per_adm = chapters.apply(len)
lines.append(f"ICD chapters per admission: mean {per_adm.mean():.1f}, median {per_adm.median():.0f}, max {per_adm.max()}")

# Service x chapter: fraction of each service's admissions with >=1 code in that chapter.
long = chapters.explode().rename("chapter").reset_index().merge(first.rename("service").reset_index())
counts = first.value_counts()
keep = counts[counts >= 5].index
table = pd.crosstab(long.service, long.chapter).loc[keep]
frac = table.div(counts.loc[keep], axis=0)
frac = frac[frac.mean().sort_values(ascending=False).index]
frac.to_csv(OUT / "mimic_service_by_chapter.csv")

# Primary-diagnosis chapter (seq_num == 1) vs service: how much does service tell us?
primary = dx[dx.seq_num == 1].set_index("hadm_id").chapter
both = pd.concat([first.rename("service"), primary.rename("chapter")], axis=1).dropna()
mi = mutual_info_score(both.service, both.chapter)
h_chapter = mutual_info_score(both.chapter, both.chapter)
lines.append(f"Primary-diagnosis chapter uncertainty explained by first service (MI / H): {mi / h_chapter:.1%}")
top = both.groupby("service").chapter.agg(lambda s: s.value_counts(normalize=True).iloc[0])
lines.append("Top-chapter purity of primary diagnosis per service (n>=5): " +
             ", ".join(f"{s} {top[s]:.0%}" for s in keep))

fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(frac.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(frac.shape[1]), frac.columns, rotation=60, ha="right")
ax.set_yticks(range(frac.shape[0]), [f"{s} (n={counts[s]})" for s in frac.index])
for i in range(frac.shape[0]):
    for j in range(frac.shape[1]):
        v = frac.values[i, j]
        if v >= 0.3:
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7, color="white" if v > 0.6 else "black")
fig.colorbar(im, label="share of admissions with >=1 code in chapter")
ax.set_title("MIMIC-IV Demo: hospital service vs ICD chapter")
fig.tight_layout()
fig.savefig(OUT / "mimic_service_by_chapter.png", dpi=150)

report = "\n".join(lines)
(OUT / "mimic_demo_summary.txt").write_text(report + "\n")
print(report)


In [ ]:
%%writefile /kaggle/working/VIP/pilot/prepare.py
"""Builds inputs for the MoE experiment:
1. Pseudo-specialty for each CodiEsp note, from a TF-IDF classifier trained on MTSamples.
2. Frozen Bio_ClinicalBERT embeddings for each CodiEsp note (512-token chunks, mean-pooled).
3. Label matrices: 3-char ICD categories, full codes, chapters.
"""
import json
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer

from common import CODIESP, DATA, OUT, SERVICES, MTSAMPLES_TO_SERVICE, CHAPTER_NAMES, icd10_chapter

ENCODER = "emilyalsentzer/Bio_ClinicalBERT"
MIN_COUNT = 3  # drop codes seen <3 times in train; the rare-code tail needs MIMIC-scale data
SPLITS = ["train", "dev", "test"]


def load_codiesp():
    docs = {}
    for s in SPLITS:
        labels = pd.read_csv(CODIESP / s / f"{s}D.tsv", sep="\t", header=None, names=["doc", "code"])
        labels = labels.groupby("doc").code.apply(lambda c: sorted(set(c.str.lower())))
        rows = []
        for path in sorted((CODIESP / s / "text_files_en").glob("*.txt")):
            rows.append({"doc": path.stem, "text": path.read_text(), "codes": labels.get(path.stem, [])})
        docs[s] = pd.DataFrame(rows)
    return docs


def pseudo_specialty(docs):
    mt = pd.read_csv(DATA / "mtsamples.csv").dropna(subset=["transcription"])
    mt["service"] = mt.medical_specialty.str.strip().map(MTSAMPLES_TO_SERVICE)
    mt = mt.dropna(subset=["service"])
    # MTSamples repeats one note under several specialties (mostly "X" + "Surgery"), which leaks
    # across splits with conflicting labels. Keep one row per text, preferring the rarest label,
    # so specific services beat the generic SURG.
    freq = mt.service.value_counts()
    mt = (mt.assign(freq=mt.service.map(freq)).sort_values("freq")
            .drop_duplicates("transcription").drop(columns="freq"))
    x_tr, x_te, y_tr, y_te = train_test_split(mt.transcription, mt.service, test_size=0.2,
                                              stratify=mt.service, random_state=0)
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2), max_features=50000, stop_words="english")
    clf = LogisticRegression(max_iter=2000, C=5, class_weight="balanced")
    clf.fit(vec.fit_transform(x_tr), y_tr)
    pred = clf.predict(vec.transform(x_te))
    stats = {"mtsamples_notes": len(mt), "heldout_accuracy": accuracy_score(y_te, pred),
             "heldout_macro_f1": f1_score(y_te, pred, average="macro"),
             "class_counts": mt.service.value_counts().to_dict()}
    assert list(clf.classes_) == SERVICES
    for s in SPLITS:
        docs[s]["specialty_probs"] = list(clf.predict_proba(vec.transform(docs[s].text)))
    dist = pd.Series(np.concatenate([docs[s].specialty_probs.apply(np.argmax) for s in SPLITS]))
    stats["codiesp_argmax_distribution"] = {SERVICES[i]: int(n) for i, n in dist.value_counts().items()}
    return stats


@torch.no_grad()
def embed(texts, device):
    tok = AutoTokenizer.from_pretrained(ENCODER)
    model = AutoModel.from_pretrained(ENCODER).to(device).eval()
    out = []
    for i, text in enumerate(texts):
        ids = tok(text, add_special_tokens=False)["input_ids"]
        chunks = [ids[j:j + 510] for j in range(0, max(len(ids), 1), 510)]
        batch = tok.pad({"input_ids": [[tok.cls_token_id] + c + [tok.sep_token_id] for c in chunks]},
                        return_tensors="pt").to(device)
        hidden = model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1)
        chunk_vecs = (hidden * mask).sum(1) / mask.sum(1)
        out.append(chunk_vecs.mean(0).cpu().numpy())
        if i % 200 == 0:
            print(f"  embedded {i}/{len(texts)}")
    return np.stack(out)


def label_matrix(code_lists, vocab):
    index = {c: i for i, c in enumerate(vocab)}
    y = np.zeros((len(code_lists), len(vocab)), dtype=np.float32)
    for r, codes in enumerate(code_lists):
        for c in codes:
            if c in index:
                y[r, index[c]] = 1
    return y


def main():
    docs = load_codiesp()
    stats = pseudo_specialty(docs)
    print(json.dumps(stats, indent=1))

    train_codes = docs["train"].codes
    full_counts = pd.Series([c for cs in train_codes for c in cs]).value_counts()
    cat_counts = pd.Series([c for cs in train_codes for c in {x.split(".")[0] for x in cs}]).value_counts()
    full_vocab = sorted(full_counts[full_counts >= MIN_COUNT].index)
    cat_vocab = sorted(cat_counts[cat_counts >= MIN_COUNT].index)
    # parent index of each full code in the category vocab (-1 if its category was filtered out)
    parent = [cat_vocab.index(c.split(".")[0]) if c.split(".")[0] in cat_vocab else -1 for c in full_vocab]
    stats.update(n_full_codes=len(full_vocab), n_categories=len(cat_vocab),
                 full_codes_with_parent=int(sum(p >= 0 for p in parent)))

    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    arrays = {"full_vocab": np.array(full_vocab), "cat_vocab": np.array(cat_vocab),
              "parent": np.array(parent), "services": np.array(SERVICES), "chapters": np.array(CHAPTER_NAMES)}
    for s in SPLITS:
        d = docs[s]
        print(f"{s}: embedding {len(d)} notes on {device}")
        arrays[f"{s}_x"] = embed(d.text.tolist(), device)
        arrays[f"{s}_full"] = label_matrix(d.codes, full_vocab)
        arrays[f"{s}_cat"] = label_matrix(d.codes.apply(lambda cs: {c.split('.')[0] for c in cs}), cat_vocab)
        arrays[f"{s}_chapter"] = label_matrix(d.codes.apply(lambda cs: {icd10_chapter(c) for c in cs}), CHAPTER_NAMES)
        arrays[f"{s}_specialty"] = np.stack(d.specialty_probs).astype(np.float32)
    np.savez_compressed(OUT / "codiesp_features.npz", **arrays)
    (OUT / "prepare_stats.json").write_text(json.dumps(stats, indent=1))
    print(json.dumps({k: v for k, v in stats.items() if k.startswith("n_") or k.startswith("full")}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/VIP/pilot/run_moe.py
"""Experiment 2: chapter-routed vs specialty-routed MoE (and stacked + parent conditioning) on CodiEsp.

Every variant uses the same frozen Bio_ClinicalBERT note embedding and predicts two code levels:
3-char ICD category ("cat") and full code ("full"). Only the routing / conditioning differs.

  flat                 one MLP (no experts)
  flat_parent          flat + child logits conditioned on parent-category logits (one model, trained jointly)
  separate_parent      two separate models: model 1 predicts categories; model 2 predicts full codes, conditioned
                       on model 1's category logits. Model 1's logits on the training notes are more confident
                       than on unseen notes, so model 2 trains on cleaner parents than it sees at test time.
  moe_learned          unsupervised gate, load-balancing loss
  moe_chapter          gate trained to predict the note's ICD chapters (chapter unknown at test time)
  moe_specialty        gate = pseudo-specialty probabilities (like MIMIC's services table, known at test time)
  moe_specialty_noMED  same, with the general-medicine (MED) expert removed
  stacked              specialty experts -> chapter experts -> FiLM(specialty, chapter) on the full-code
                       level + parent conditioning
  stacked_noparent     stacked without parent conditioning
  stacked_nofilm       stacked without FiLM
"""
import argparse
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from common import OUT

HIDDEN = 128
VARIANTS = ["flat", "flat_parent", "separate_parent", "moe_learned", "moe_chapter", "moe_specialty", "moe_specialty_noMED",
            "stacked", "stacked_noparent", "stacked_nofilm"]


def experts(n, d_in, d_out):
    return nn.ModuleList(nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU()) for _ in range(n))


def mix(expert_list, gate, x):
    return torch.einsum("be,beh->bh", gate, torch.stack([e(x) for e in expert_list], 1))


class Model(nn.Module):
    def __init__(self, variant, d_in, n_cat, n_full, parent, n_chapters, n_specialties):
        super().__init__()
        self.v = variant
        self.drop = nn.Dropout(0.2)
        self.register_buffer("parent", torch.as_tensor(parent))
        stacked = variant.startswith("stacked")
        if variant.startswith("flat"):
            self.body = nn.Sequential(nn.Linear(d_in, 4 * HIDDEN), nn.ReLU(), nn.Linear(4 * HIDDEN, HIDDEN), nn.ReLU())
        elif variant == "moe_learned":
            self.gate, self.ex = nn.Linear(d_in, 8), experts(8, d_in, HIDDEN)
        elif variant == "moe_chapter":
            self.gate, self.ex = nn.Linear(d_in, n_chapters), experts(n_chapters, d_in, HIDDEN)
        elif variant.startswith("moe_specialty"):
            self.ex = experts(n_specialties, d_in, HIDDEN)
        if stacked:
            self.spec_ex = experts(n_specialties, d_in, HIDDEN)
            self.chap_gate = nn.Linear(d_in + HIDDEN, n_chapters)
            self.chap_ex = experts(n_chapters, HIDDEN, HIDDEN)
        self.use_film = stacked and variant != "stacked_nofilm"
        self.use_parent = variant in ("flat_parent", "stacked", "stacked_nofilm")
        if self.use_film:
            self.film = nn.Linear(n_specialties + n_chapters, 2 * HIDDEN)
            nn.init.zeros_(self.film.weight), nn.init.zeros_(self.film.bias)
        if self.use_parent:
            self.parent_w = nn.Parameter(torch.full((n_full,), 0.5))
        self.cat_head, self.full_head = nn.Linear(HIDDEN, n_cat), nn.Linear(HIDDEN, n_full)

    def forward(self, x, spec, parent_logits=None):
        x = self.drop(x)
        gate = None
        if self.v.startswith("flat"):
            z = self.body(x)
        elif self.v.startswith("moe_specialty"):
            gate = spec
            z = mix(self.ex, gate, x)
        elif self.v.startswith("moe"):
            gate = F.softmax(self.gate(x), -1)
            z = mix(self.ex, gate, x)
        else:
            z_s = mix(self.spec_ex, spec, x)
            gate = F.softmax(self.chap_gate(torch.cat([x, z_s], -1)), -1)
            z = z_s + mix(self.chap_ex, gate, z_s)
        cat = self.cat_head(z)
        z_full = z
        if self.use_film:
            gamma, beta = self.film(torch.cat([spec, gate], -1)).chunk(2, -1)
            z_full = z * (1 + gamma) + beta
        full = self.full_head(z_full)
        if self.use_parent:
            parents = cat if parent_logits is None else parent_logits
            full = full + self.parent_w * parents[:, self.parent]
        return cat, full, gate


def best_threshold_f1(probs, y):
    """Micro-F1 maximised over one global threshold. Returns (threshold, micro_f1)."""
    best = (0.5, 0.0)
    for t in np.arange(0.05, 0.95, 0.025):
        p = probs >= t
        tp = (p & (y == 1)).sum()
        f1 = 2 * tp / max(p.sum() + y.sum(), 1)
        best = max(best, (t, f1), key=lambda b: b[1])
    return best


def f1_scores(probs, y, t):
    p, y = probs >= t, y == 1
    tp, fp, fn = (p & y).sum(0), (p & ~y).sum(0), (~p & y).sum(0)
    micro = 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)
    present = y.sum(0) > 0  # macro over labels with >=1 positive in this split
    macro = (2 * tp / np.maximum(2 * tp + fp + fn, 1))[present].mean()
    return micro, macro


def specialty_inputs(variant, data):
    """Per-split specialty routing probabilities; the noMED variant drops MED and renormalises."""
    spec = {s: torch.as_tensor(data[f"{s}_specialty"]) for s in ["train", "dev", "test"]}
    if variant == "moe_specialty_noMED":
        keep = torch.as_tensor([s != "MED" for s in data["services"]])
        spec = {k: v[:, keep] / v[:, keep].sum(1, keepdim=True) for k, v in spec.items()}
    return spec


def compute_loss(variant, cat, full, gate, y_cat, y_full, y_chapter):
    loss = F.binary_cross_entropy_with_logits(cat, y_cat) + F.binary_cross_entropy_with_logits(full, y_full)
    if variant == "moe_chapter" or variant.startswith("stacked"):
        target = y_chapter / y_chapter.sum(1, keepdim=True).clamp(min=1)
        loss = loss + 0.1 * -(target * torch.log(gate + 1e-8)).sum(1).mean()
    if variant == "moe_learned":
        loss = loss + 0.01 * gate.shape[1] * (gate.mean(0) ** 2).sum()
    return loss


def evaluate(variant, data, dev_probs, test_probs, test_gate, seed, params):
    """Tunes one threshold per level on dev, reports test F1 and routing stats."""
    tc = best_threshold_f1(dev_probs[0], data["dev_cat"])[0]
    tf = best_threshold_f1(dev_probs[1], data["dev_full"])[0]
    row = {"variant": variant, "seed": seed, "params": params}
    row["cat_micro"], row["cat_macro"] = f1_scores(test_probs[0], data["test_cat"], tc)
    row["full_micro"], row["full_macro"] = f1_scores(test_probs[1], data["test_full"], tf)
    if test_gate is not None:
        g = test_gate
        row["gate_entropy"] = float(-(g * np.log(g + 1e-8)).sum(1).mean() / np.log(g.shape[1]))
        row["experts_used_80pct"] = float(np.mean((np.sort(g, 1)[:, ::-1].cumsum(1) < 0.8).sum(1) + 1))
    return row


def fit(model, train_step, dev_score, epochs=400):
    """Full-batch AdamW; keeps the weights with the best dev score (checked every 10 epochs)."""
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    best, best_state = -1, None
    for epoch in range(epochs):
        loss = train_step()
        opt.zero_grad(), loss.backward(), opt.step()
        if epoch % 10 == 9:
            model.eval()
            with torch.no_grad():
                score = dev_score()
            model.train()
            if score > best:
                best, best_state = score, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()


def run_separate(data, seed, epochs=400):
    """Two independently trained models: categories first, then full codes conditioned on them."""
    torch.manual_seed(seed), np.random.seed(seed)
    t = {k: torch.as_tensor(v) for k, v in data.items() if v.dtype.kind in "fiub"}
    spec = specialty_inputs("flat", data)
    dims = (t["train_x"].shape[1], t["train_cat"].shape[1], t["train_full"].shape[1], data["parent"],
            t["train_chapter"].shape[1], spec["train"].shape[1])
    first, second = Model("flat", *dims), Model("flat_parent", *dims)

    fit(first, lambda: F.binary_cross_entropy_with_logits(first(t["train_x"], spec["train"])[0], t["train_cat"]),
        lambda: best_threshold_f1(torch.sigmoid(first(t["dev_x"], spec["dev"])[0]).numpy(), data["dev_cat"])[1], epochs)
    with torch.no_grad():
        cat_logits = {s: first(t[f"{s}_x"], spec[s])[0] for s in ["train", "dev", "test"]}

    fit(second, lambda: F.binary_cross_entropy_with_logits(
            second(t["train_x"], spec["train"], cat_logits["train"])[1], t["train_full"]),
        lambda: best_threshold_f1(torch.sigmoid(second(t["dev_x"], spec["dev"], cat_logits["dev"])[1]).numpy(),
                                  data["dev_full"])[1], epochs)
    with torch.no_grad():
        probs = {s: (torch.sigmoid(cat_logits[s]).numpy(),
                     torch.sigmoid(second(t[f"{s}_x"], spec[s], cat_logits[s])[1]).numpy(), None) for s in ["dev", "test"]}
    params = sum(p.numel() for p in first.parameters()) + sum(p.numel() for p in second.parameters())
    return evaluate("separate_parent", data, probs["dev"], probs["test"], None, seed, params)


def run(variant, data, seed, epochs=400):
    torch.manual_seed(seed), np.random.seed(seed)
    t = {k: torch.as_tensor(v) for k, v in data.items() if v.dtype.kind in "fiub"}
    spec = specialty_inputs(variant, data)
    model = Model(variant, t["train_x"].shape[1], t["train_cat"].shape[1], t["train_full"].shape[1],
                  data["parent"], t["train_chapter"].shape[1], spec["train"].shape[1])

    def predict(split):
        with torch.no_grad():
            cat, full, gate = model(t[f"{split}_x"], spec[split])
        return torch.sigmoid(cat).numpy(), torch.sigmoid(full).numpy(), None if gate is None else gate.numpy()

    def train_step():
        cat, full, gate = model(t["train_x"], spec["train"])
        return compute_loss(variant, cat, full, gate, t["train_cat"], t["train_full"], t["train_chapter"])

    def dev_score():
        pc, pf, _ = predict("dev")
        return best_threshold_f1(pc, data["dev_cat"])[1] + best_threshold_f1(pf, data["dev_full"])[1]

    fit(model, train_step, dev_score, epochs)
    dev, test = predict("dev"), predict("test")
    return evaluate(variant, data, dev, test, test[2], seed, sum(p.numel() for p in model.parameters()))


def report(df, name, title):
    """Writes <name>.md (mean ± std table) and <name>.png from per-seed rows."""
    metrics = ["cat_micro", "cat_macro", "full_micro", "full_macro"]
    agg = df.groupby("variant", sort=False)[metrics].agg(["mean", "std"])
    extra = df.groupby("variant", sort=False)[["params", "gate_entropy", "experts_used_80pct"]].mean()
    table = pd.DataFrame({m: [f"{agg.loc[v, (m, 'mean')]:.3f} ± {agg.loc[v, (m, 'std')]:.3f}" for v in agg.index]
                          for m in metrics}, index=agg.index)
    table["params"] = extra.params.map(lambda p: f"{p / 1e3:.0f}k")
    table["gate entropy"] = extra.gate_entropy.map(lambda e: "" if pd.isna(e) else f"{e:.2f}")
    table["experts for 80% weight"] = extra.experts_used_80pct.map(lambda e: "" if pd.isna(e) else f"{e:.1f}")
    md = table.to_markdown()
    (OUT / f"{name}.md").write_text(md + "\n")
    print(md)

    n_seeds = df.seed.nunique()
    fig, axes = plt.subplots(1, 2, figsize=(12, 0.5 * len(agg) + 2), sharey=False)
    for ax, level in zip(axes, ["cat", "full"]):
        for i, kind in enumerate(["micro", "macro"]):
            m = agg[(f"{level}_{kind}", "mean")]
            s = agg[(f"{level}_{kind}", "std")]
            ax.barh(np.arange(len(m)) + i * 0.4, m, 0.4, xerr=s, label=f"{kind}-F1", capsize=2)
        ax.set_yticks(np.arange(len(agg)) + 0.2, agg.index)
        ax.invert_yaxis()
        ax.set_title({"cat": "3-char category", "full": "Full ICD-10 code"}[level])
        ax.set_xlabel(f"test F1 (mean ± std, {n_seeds} seeds)")
    fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=2)
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--plot-only", action="store_true", help="redraw from the saved per-seed CSV")
    args = ap.parse_args()
    title = "CodiEsp (English) pilot, frozen encoder: routing strategy vs ICD coding F1"
    if args.plot_only:
        return report(pd.read_csv(OUT / "moe_results_by_seed.csv"), "moe_results", title)
    data = dict(np.load(OUT / "codiesp_features.npz", allow_pickle=True))
    mu, sd = data["train_x"].mean(0), data["train_x"].std(0) + 1e-6
    for s in ["train", "dev", "test"]:
        data[f"{s}_x"] = ((data[f"{s}_x"] - mu) / sd).astype(np.float32)
    torch.set_num_threads(8)

    rows = []
    for v in VARIANTS:
        for s in range(args.seeds):
            rows.append(run_separate(data, s) if v == "separate_parent" else run(v, data, s))
        r = pd.DataFrame(rows[-args.seeds:])
        print(f"{v:22s} cat micro {r.cat_micro.mean():.3f}  full micro {r.full_micro.mean():.3f}  "
              f"full macro {r.full_macro.mean():.3f}")
    df = pd.DataFrame(rows)
    df.to_csv(OUT / "moe_results_by_seed.csv", index=False)
    report(df, "moe_results", title)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/VIP/pilot/finetune.py
"""Experiment 2b (GPU): same routing variants as run_moe.py, but Bio_ClinicalBERT is fine-tuned end to end
instead of frozen. Meant for a free Kaggle/Colab GPU (tested only on a small sample; runtime not yet measured).
Requires prepare.py to have run (labels + pseudo-specialty come from codiesp_features.npz).
"""
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoModel, AutoTokenizer
from common import OUT
from prepare import ENCODER, SPLITS, load_codiesp
from run_moe import Model, best_threshold_f1, compute_loss, evaluate, report, specialty_inputs

MAX_CHUNKS = 3  # 3 x 510 tokens covers ~99% of CodiEsp notes (99th percentile ~1.2k tokens)
DEFAULT_VARIANTS = ["flat", "flat_parent", "moe_chapter", "moe_specialty", "stacked"]


class Coder(nn.Module):
    def __init__(self, variant, data, n_specialties):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(ENCODER)
        d = self.encoder.config.hidden_size
        self.norm = nn.LayerNorm(d)
        self.head = Model(variant, d, data["train_cat"].shape[1], data["train_full"].shape[1], data["parent"],
                          data["train_chapter"].shape[1], n_specialties)

    def forward(self, chunks, doc_index, n_docs, spec):
        h = self.encoder(**chunks).last_hidden_state
        mask = chunks["attention_mask"].unsqueeze(-1)
        chunk_vecs = ((h * mask).sum(1) / mask.sum(1)).float()
        doc = torch.zeros(n_docs, h.shape[-1], device=h.device).index_add_(0, doc_index, chunk_vecs)
        doc = doc / torch.bincount(doc_index, minlength=n_docs).unsqueeze(1)
        # heads stay fp32: the gate loss takes log(gate), which underflows in fp16
        with torch.autocast(h.device.type, enabled=False):
            return self.head(self.norm(doc), spec)


def tokenize(tok, texts):
    out = []
    for text in texts:
        ids = tok(text, add_special_tokens=False)["input_ids"][: 510 * MAX_CHUNKS]
        out.append([[tok.cls_token_id] + ids[j:j + 510] + [tok.sep_token_id] for j in range(0, max(len(ids), 1), 510)])
    return out


def make_batch(tok, doc_tokens, idx, device):
    chunks, doc_index = [], []
    for k, i in enumerate(idx):
        chunks += doc_tokens[i]
        doc_index += [k] * len(doc_tokens[i])
    return tok.pad({"input_ids": chunks}, return_tensors="pt").to(device), torch.tensor(doc_index, device=device)


def run(variant, data, tokens, tok, seed, epochs, batch_size, device):
    torch.manual_seed(seed), np.random.seed(seed)
    spec = {k: v.to(device) for k, v in specialty_inputs(variant, data).items()}
    model = Coder(variant, data, spec["train"].shape[1]).to(device)
    head_params = list(model.norm.parameters()) + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.encoder.parameters(), "lr": 2e-5},
                             {"params": head_params, "lr": 1e-3}], weight_decay=1e-2)
    y = {k: torch.as_tensor(data[f"train_{k}"]).to(device) for k in ["cat", "full", "chapter"]}
    amp = device == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp)

    def predict(split):
        model.eval()
        cats, fulls, gates = [], [], []
        with torch.no_grad(), torch.autocast(device, enabled=amp):
            for s in range(0, len(tokens[split]), 2 * batch_size):
                idx = list(range(s, min(s + 2 * batch_size, len(tokens[split]))))
                chunks, di = make_batch(tok, tokens[split], idx, device)
                cat, full, gate = model(chunks, di, len(idx), spec[split][idx])
                cats.append(torch.sigmoid(cat).cpu()), fulls.append(torch.sigmoid(full).cpu())
                gates.append(None if gate is None else gate.cpu())
        model.train()
        return torch.cat(cats).numpy(), torch.cat(fulls).numpy(), None if gates[0] is None else torch.cat(gates).numpy()

    best, best_state = -1, None
    for epoch in range(epochs):
        perm = np.random.permutation(len(tokens["train"]))
        for s in range(0, len(perm), batch_size):
            idx = perm[s:s + batch_size].tolist()
            chunks, di = make_batch(tok, tokens["train"], idx, device)
            with torch.autocast(device, enabled=amp):
                cat, full, gate = model(chunks, di, len(idx), spec["train"][idx])
            loss = compute_loss(variant, cat, full, gate, y["cat"][idx], y["full"][idx], y["chapter"][idx])
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        pc, pf, _ = predict("dev")
        score = best_threshold_f1(pc, data["dev_cat"])[1] + best_threshold_f1(pf, data["dev_full"])[1]
        print(f"  {variant} seed {seed} epoch {epoch + 1}: loss {loss.item():.4f}  dev cat+full micro-F1 {score:.3f}")
        if score > best:
            best, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    dev, test = predict("dev"), predict("test")
    return evaluate(variant, data, dev, test, test[2], seed, sum(p.numel() for p in head_params))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variants", nargs="+", default=DEFAULT_VARIANTS)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=4, help="notes per step (each up to 3 chunks)")
    ap.add_argument("--smoke", action="store_true", help="16 notes per split, 1 epoch: checks the code runs")
    args = ap.parse_args()
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

    data = dict(np.load(OUT / "codiesp_features.npz", allow_pickle=True))
    tok = AutoTokenizer.from_pretrained(ENCODER)
    docs = load_codiesp()  # same sorted order prepare.py used to build the npz
    tokens = {s: tokenize(tok, docs[s].text) for s in SPLITS}
    for s in SPLITS:
        assert len(tokens[s]) == len(data[f"{s}_cat"])
    name = "finetune_results"
    if args.smoke:
        name, args.epochs, args.seeds = "finetune_smoke", 1, 1
        for s in SPLITS:
            tokens[s] = tokens[s][:16]
            for k in ["x", "cat", "full", "chapter", "specialty"]:
                data[f"{s}_{k}"] = data[f"{s}_{k}"][:16]

    rows = []
    for v in args.variants:
        for seed in range(args.seeds):
            rows.append(run(v, data, tokens, tok, seed, args.epochs, args.batch_size, device))
            pd.DataFrame(rows).to_csv(OUT / f"{name}_by_seed.csv", index=False)  # survives a killed session
            print(rows[-1])
    report(pd.DataFrame(rows), name, "CodiEsp (English), fine-tuned Bio_ClinicalBERT: routing strategy vs ICD coding F1")


if __name__ == "__main__":
    main()


In [ ]:
%cd /kaggle/working/VIP/pilot
!python common.py && python mimic_demo.py

In [ ]:
!python prepare.py 2>&1 | grep -v -E 'UNEXPECTED|LOAD REPORT'

In [ ]:
!python run_moe.py --seeds 5

## Fine-tuning (slow part)
Saves per-seed results after every run, so a killed session keeps what finished. To cut time: `--seeds 2` or `--epochs 6`.

In [ ]:
!python finetune.py --seeds 3 --epochs 10 2>&1 | grep -v -E 'UNEXPECTED|LOAD REPORT'

In [ ]:
from IPython.display import Image, Markdown, display
for name in ['mimic_service_by_chapter', 'moe_results', 'finetune_results']:
    display(Image(f'out/{name}.png'))
for name in ['moe_results', 'finetune_results']:
    display(Markdown(open(f'out/{name}.md').read()))
print(open('out/mimic_demo_summary.txt').read())

In [ ]:
!cd /kaggle/working && zip -qr pilot_results.zip VIP/pilot/out && ls -la pilot_results.zip